## Install Dependencies

In [3]:
!pip install opencv-python matplotlib

In [5]:
#Only for macos
!pip install tensorflow-macos tensorflow-metal

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.0/227.0 MB 1.1 MB/s eta 0:00:0000:0100:06
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 1.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.8/389.8 kB 1.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 1.3 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: ml-dtypes
    Found existing installation: ml-dtypes 0.4.1
    Uninstalling ml-dtypes-0.4.1:
      Successfully uninstalled ml-dtypes-0.4.1
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.17.1
    Uninstalling tensorboard-2.17.1:
      Successfully uninstalled tensorboard-2.17.1
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.17.0
    Uninstalling tensorflow-2.17.0:
      Successfully uninstalled tensorflow-2.17.0


## Import Dependencies

In [2]:
#Import standard dependencies
import cv2
import os
import random
import numpy as np
import matplotlib.pyplot as plt

In [3]:
# Import tensorflow dependencies - Functional API
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Layer, Conv2D, Dense, MaxPooling2D, Input, Flatten

## GPU Growth

In [4]:
# Avoid OOM errors by setting GPU Memory Consumption Growth
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

## Create Folder Structures

In [5]:
#Setup Paths
POS_PATH = os.path.join('data', 'positive')
NEG_PATH = os.path.join('data', 'negative')
ANC_PATH = os.path.join('data', 'anchor')

In [ ]:
# Make the directories
os.makedirs(POS_PATH)
os.makedirs(NEG_PATH)
os.makedirs(ANC_PATH)

## LFW Dataset

In [6]:
#Move LFW images to data/negative folder
for directory in os.listdir('lfw'):
    if directory.startswith('.'):
        continue  
    dir_path = os.path.join('lfw', directory)
    for file in os.listdir(dir_path):
        EX_PATH = os.path.join('lfw', directory, file)
        NEW_PATH = os.path.join(NEG_PATH, file)
        os.replace(EX_PATH, NEW_PATH)

## Collect Positive Anchor Classes

In [7]:
#Import uuid for generating unique names
import uuid

In [8]:
#Webcam connection establishment
cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    
    #Cut down frame to 250X250
    frame = frame[250:250+250,530:530+250,:]
    
    #Collect Anchor image
    if cv2.waitKey(1) & 0XFF==ord('a'):
        imgname = os.path.join(ANC_PATH, '{}.jpg'.format(uuid.uuid1()))
        cv2.imwrite(imgname,frame)
        
    #Collect Positives
    if cv2.waitKey(1) & 0XFF==ord('p'):
        imgname = os.path.join(POS_PATH, '{}.jpg'.format(uuid.uuid1()))
        cv2.imwrite(imgname,frame)
        
    #Show image to screen
    cv2.imshow('Image Collection', frame)
    
    #Breaking 
    if cv2.waitKey(1) & 0XFF==ord('q'):
        break
        
#Release Webcam
cap.release()
#Close image show frame
if cv2.getWindowProperty('Image Collection', cv2.WND_PROP_VISIBLE) >= 1:
    cv2.destroyWindow('Image Collection')
cv2.waitKey(100)
cv2.destroyAllWindows()

## Load and Preprocess Images

In [10]:
anchor = tf.data.Dataset.list_files(ANC_PATH + '/*.jpg')
positive = tf.data.Dataset.list_files(POS_PATH + '/*.jpg')
negative = tf.data.Dataset.list_files(NEG_PATH + '/*.jpg')

In [13]:
dir_test = anchor.as_numpy_iterator()

In [14]:
print(dir.next())

b'data/anchor/2b4dc562-1c97-11f0-ba32-9e47e35cc52b.jpg'


In [11]:
def preprocess(file_path):
    byte_img = tf.io.read_file(file_path)
    img = tf.io.decode_jpeg(byte_img)
    img = tf.image.resize(img, (100, 100))
    img = img / 255.0
    return img

In [16]:
trial_img = preprocess('data/anchor/2b4dc562-1c97-11f0-ba32-9e47e35cc52b.jpg')

In [19]:
anchor = anchor.map(preprocess)
positive = positive.map(preprocess)
negative = negative.map(preprocess)

In [21]:
positives = tf.data.Dataset.zip((anchor, positive, tf.data.Dataset.from_tensor_slices(tf.ones(len(anchor)))))
negatives = tf.data.Dataset.zip((anchor, negative, tf.data.Dataset.from_tensor_slices(tf.zeros(len(anchor)))))
data = positives.concatenate(negatives)

In [22]:
data

<_ConcatenateDataset element_spec=(TensorSpec(shape=(100, 100, None), dtype=tf.float32, name=None), TensorSpec(shape=(100, 100, None), dtype=tf.float32, name=None), TensorSpec(shape=(), dtype=tf.float32, name=None))>

In [23]:
samples = data.as_numpy_iterator()

In [24]:
samples.next()

(array([[[0.51740193, 0.49387255, 0.5330882 ],
         [0.51053923, 0.4870098 , 0.52818626],
         [0.5144608 , 0.49093136, 0.5379902 ],
         ...,
         [0.7107843 , 0.75392157, 0.7617647 ],
         [0.7053922 , 0.74852943, 0.7563726 ],
         [0.7117647 , 0.75490195, 0.7627451 ]],
 
        [[0.502451  , 0.48186275, 0.52009803],
         [0.502451  , 0.48186275, 0.52205884],
         [0.5115196 , 0.49093136, 0.5370098 ],
         ...,
         [0.69411767, 0.7362745 , 0.7441176 ],
         [0.7007353 , 0.74289215, 0.7507353 ],
         [0.7142157 , 0.7563726 , 0.7642157 ]],
 
        [[0.49730393, 0.48553923, 0.5208333 ],
         [0.51617646, 0.50441176, 0.5416667 ],
         [0.5034314 , 0.49166667, 0.5348039 ],
         ...,
         [0.7139706 , 0.7531863 , 0.7610294 ],
         [0.7078431 , 0.7470588 , 0.75490195],
         [0.70612746, 0.74534315, 0.7531863 ]],
 
        ...,
 
        [[0.70098037, 0.68921566, 0.63039213],
         [0.65392154, 0.6392157 , 0.58137